# Proyecto de Análisis de Series de Tiempo: AirPassengers

Este notebook consolida todo el flujo de trabajo: Descarga de datos, Análisis Exploratorio (EDA) y Modelado con SARIMA.

## 1. Descarga del Dataset

In [ ]:
import kagglehub
import shutil
import os

In [ ]:
print("Iniciando la descarga del dataset 'Air Passengers'...")

In [ ]:
# 1. Descargar la última versión desde Kaggle (se descarga en una carpeta caché del sistema)
path_cache = kagglehub.dataset_download("chirag19/air-passengers")

In [ ]:
print(f"Dataset descargado en caché: {path_cache}")

In [ ]:
# 2. Encontrar el archivo CSV dentro de esa carpeta
archivo_csv = "AirPassengers.csv" # Nombre clásico de este archivo en Kaggle
ruta_origen = os.path.join(path_cache, archivo_csv)

In [ ]:
# 3. Moverlo a nuestra carpeta de trabajo actual
ruta_destino = os.path.join(os.getcwd(), archivo_csv)

In [ ]:
if os.path.exists(ruta_origen):
    shutil.copy(ruta_origen, ruta_destino)
    print(f"\n¡Éxito! El archivo se ha copiado a tu carpeta actual:")
    print(f"-> {ruta_destino}")
else:
    print(f"\nNo se encontró el archivo {archivo_csv} en la ruta descargada.")
    print("Revisa la carpeta caché manualmente.")

## 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
import pandas as pd

In [ ]:
print("--- INICIANDO EDA: AIR PASSENGERS ---")

In [ ]:
# 1. Cargar el dataset
try:
    df = pd.read_csv("AirPassengers.csv")
    print("\nDataset cargado exitosamente.")
except FileNotFoundError:
    print("Error: No se encontró 'AirPassengers.csv'. Asegúrate de haber ejecutado el script 01.")
    exit()

In [ ]:
# 2. Vistazo general
print("\n--- PRIMERAS 5 FILAS ---")
print(df.head())

In [ ]:
print("\n--- INFORMACIÓN GENERAL ---")
df.info()

In [ ]:
# 3. Revisión de Nulos y Duplicados
print("\n--- REVISIÓN DE NULOS ---")
nulos = df.isnull().sum()
print(nulos)

In [ ]:
if nulos.sum() == 0:
    print("-> ¡Excelente! El dataset está limpio, no hay valores nulos.")
else:
    print("-> ¡Atención! Se encontraron valores nulos.")

In [ ]:
print("\n--- REVISIÓN DE DUPLICADOS ---")
duplicados = df.duplicated().sum()
print(f"Filas duplicadas encontradas: {duplicados}")

In [ ]:
# 4. Preparación clave para Series de Tiempo
print("\n--- TRANSFORMACIÓN DE FECHAS ---")

In [ ]:
# Renombramos las columnas para que no tengan caracteres raros como '#'
df.rename(columns={'Month': 'fecha', '#Passengers': 'pasajeros'}, inplace=True)

In [ ]:
# Convertimos el texto (ej: '1949-01') a un formato de tiempo matemático
df['fecha'] = pd.to_datetime(df['fecha'])

In [ ]:
# En Series de Tiempo, la fecha SIEMPRE debe ser el índice del DataFrame
df.set_index('fecha', inplace=True)

In [ ]:
print("\n--- ESTRUCTURA FINAL (LISTA PARA MODELOS) ---")
print(df.head())
print(f"\nTipo de índice actual: {type(df.index)}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
print("\n--- EDA ESPECÍFICO PARA SERIES DE TIEMPO ---")

In [ ]:
# 1. Gráfico principal (Para ver Tendencia a ojo)
plt.figure(figsize=(10, 5))
plt.plot(df.index, df['pasajeros'], color='blue')
plt.title('Evolución de Pasajeros de Aerolíneas (1949-1960)')
plt.xlabel('Fecha')
plt.ylabel('Cantidad de Pasajeros')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 2. Análisis de "Outliers" Estacionales (Boxplot por Mes)
# Queremos ver si los meses de verano siempre tienen picos o si hubo un verano atípico.
df['mes'] = df.index.month
plt.figure(figsize=(10, 5))
sns.boxplot(x='mes', y='pasajeros', data=df, palette='viridis')
plt.title('Distribución de pasajeros por mes (Patrón Estacional)')
plt.xlabel('Mes del año')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# 3. Descomposición de la Serie de Tiempo
# Magia matemática: separa la gráfica en Tendencia, Estacionalidad y Residuos (Ruido)
print("\nGenerando gráfica de descomposición. Cierra la ventana del gráfico anterior para ver esta.")
descomposicion = seasonal_decompose(df['pasajeros'], model='multiplicative')
fig = descomposicion.plot()
fig.set_size_inches(10, 8)
plt.show()

In [ ]:
# Limpiamos la columna extra que creamos solo para graficar
df.drop(columns=['mes'], inplace=True)

## 3. Preparación y Modelado (SARIMA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import warnings

In [ ]:
# Ignorar warnings menores de statsmodels para tener una consola más limpia
warnings.filterwarnings("ignore")

In [ ]:
print("--- 03: PREPARACIÓN Y MODELADO (SARIMA) ---")

In [ ]:
# 1. Cargar datos
print("\nCargando datos limpios...")
df = pd.read_csv("AirPassengers.csv")
df.rename(columns={'Month': 'fecha', '#Passengers': 'pasajeros'}, inplace=True)
df['fecha'] = pd.to_datetime(df['fecha'])
df.set_index('fecha', inplace=True)

In [ ]:
# 2. Prueba de Estacionariedad (Dickey-Fuller)
print("\n--- PRUEBA DE ESTACIONARIEDAD (Dickey-Fuller) ---")
# Una serie es estacionaria si sus propiedades estadísticas (media, varianza) son constantes en el tiempo.
resultado_adf = adfuller(df['pasajeros'])
print(f"Estadístico ADF: {resultado_adf[0]:.4f}")
print(f"Valor p (p-value): {resultado_adf[1]:.4f}")
if resultado_adf[1] < 0.05:
    print("-> La serie es ESTACIONARIA (podemos modelar directamente).")
else:
    print("-> La serie NO es estacionaria (hay tendencia/estacionalidad). SARIMA se encargará de esto aplicando diferenciación internamente.")

In [ ]:
# 3. División en Train y Test (Cronológica)
print("\n--- DIVISIÓN DE DATOS (TRAIN / TEST) ---")
# Dejamos los últimos 2 años (24 meses) para Test y el resto para Train
train = df.iloc[:-24]
test = df.iloc[-24:]

In [ ]:
print(f"Datos de Entrenamiento (Train): {len(train)} meses")
print(f"Datos de Prueba (Test): {len(test)} meses")

In [ ]:
# 4. Entrenamiento del Modelo SARIMA
print("\n--- ENTRENANDO MODELO SARIMA ---")
# SARIMA necesita dos conjuntos de parámetros:
# order = (p, d, q) -> Componentes No Estacionales (Autoregresivo, Diferenciación, Media Móvil)
# seasonal_order = (P, D, Q, s) -> Componentes Estacionales. "s=12" porque hay ciclos anuales (12 meses).
modelo = SARIMAX(train['pasajeros'], 
                 order=(1, 1, 1), 
                 seasonal_order=(1, 1, 1, 12),
                 enforce_stationarity=False,
                 enforce_invertibility=False)

In [ ]:
resultado_modelo = modelo.fit(disp=False)
print("¡Modelo entrenado exitosamente!")
print("\nResumen de los coeficientes del modelo:")
print(resultado_modelo.summary().tables[1])

In [ ]:
# 5. Predicciones
print("\n--- REALIZANDO PREDICCIONES ---")
# Le pedimos al modelo que prediga desde el final de Train hasta el final de Test
predicciones = resultado_modelo.predict(start=len(train), end=len(train)+len(test)-1, dynamic=False)
predicciones.index = test.index

In [ ]:
# Calcular Métricas de Error
rmse = np.sqrt(mean_squared_error(test['pasajeros'], predicciones))
mae = mean_absolute_error(test['pasajeros'], predicciones)
print(f"\nError Cuadrático Medio (RMSE): {rmse:.2f}")
print(f"Error Absoluto Medio (MAE): {mae:.2f}")

In [ ]:
# 6. Gráfico Final (Real vs Predicción)
print("\n--- GENERANDO GRÁFICO FINAL ---")
plt.figure(figsize=(12, 6))
plt.plot(train.index, train['pasajeros'], label='Entrenamiento (Train)', color='blue')
plt.plot(test.index, test['pasajeros'], label='Prueba Real (Test)', color='green')
plt.plot(predicciones.index, predicciones, label='Predicciones SARIMA', color='red', linestyle='--')
plt.title('Predicción de Pasajeros de Aerolíneas usando SARIMA')
plt.xlabel('Fecha')
plt.ylabel('Cantidad de Pasajeros')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()